In [ ]:
import os
import subprocess
import sys
from datetime import datetime

print(f"Initial working directory: {os.getcwd()}")

# Find project root by looking for config folder
# Search up to 3 levels up
found = False
for _ in range(4):
    if os.path.exists('config') and os.path.exists('template.ipynb'):
        found = True
        break
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():  # Already at root
        break
    os.chdir(parent)

if not found:
    raise FileNotFoundError(f"Cannot find project root (config folder not found). Current dir: {os.getcwd()}")

print(f"Project root: {os.getcwd()}")

In [ ]:
def find_configs():
    configs = []
    for root, dirs, files in os.walk('config'):
        if 'config.yaml' in files:
            configs.append(root)
    return sorted(configs)

configs = find_configs()
print(f"Found {len(configs)} experiment(s):")
for c in configs:
    print(f"  - {c}")

In [ ]:
def run_experiment(config_dir):
    """
    Run a single experiment using papermill in a SUBPROCESS.
    
    This ensures complete memory isolation between runs - when the subprocess
    exits, ALL memory from that experiment is released back to the OS.
    
    This fixes the memory leak where pm.execute_notebook() keeps references
    to executed notebook variables in the parent kernel's memory.
    """
    config_path = os.path.join(config_dir, 'config.yaml')
    output_dir = config_dir.replace('config/', 'output/', 1)
    os.makedirs(output_dir, exist_ok=True)
    output_notebook = os.path.join(output_dir, 'notebook.ipynb')
    
    # Run papermill as subprocess for memory isolation
    # Use py39_26v1 kernel to ensure correct SHAP version
    cmd = [
        sys.executable, '-m', 'papermill',
        'template.ipynb',
        output_notebook,
        '-p', 'config_path', config_path,
        '-k', 'py39_26v1'
    ]
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=3600  # 1 hour timeout per experiment
        )
        if result.returncode == 0:
            return True, output_notebook, None
        else:
            # Get last 500 chars of stderr for error message
            error_msg = result.stderr[-500:] if result.stderr else "Unknown error"
            return False, output_notebook, error_msg
    except subprocess.TimeoutExpired:
        return False, output_notebook, "Timeout after 1 hour"
    except Exception as e:
        return False, output_notebook, str(e)

In [ ]:
results = []
start = datetime.now()

for i, config_dir in enumerate(configs, 1):
    exp_start = datetime.now()
    print(f"[{i}/{len(configs)}] {config_dir}")
    
    success, output, error = run_experiment(config_dir)
    
    exp_elapsed = datetime.now() - exp_start
    results.append({
        'config': config_dir, 
        'success': success, 
        'output': output, 
        'error': error,
        'elapsed': exp_elapsed
    })
    print(f"  {'✓' if success else '✗'} {output} ({exp_elapsed})")

elapsed = datetime.now() - start
successful = sum(1 for r in results if r['success'])

print(f"\nCompleted in {elapsed}")
print(f"Results: {successful}/{len(configs)} successful")

In [ ]:
failed = [r for r in results if not r['success']]
if failed:
    print("Failed experiments:")
    for r in failed:
        print(f"\n  - {r['config']}:")
        print(f"    {r['error'][:200] if r['error'] else 'No error message'}")